#Imports de librerias

In [0]:
from pyspark.sql.functions import col, lower, upper, trim, initcap

#Lectura de la Tabla Bronce olist_sellers

In [0]:
df = spark.table("`catalog_brazilian-e-commerce`.bronze.olist_sellers_dataset")

In [0]:
df.display()

# Transformaciones

In [0]:
df = (
    df
    
    # Limpieza de strings
    .withColumn("seller_city", trim(lower(col("seller_city"))))
    .withColumn("seller_state", trim(upper(col("seller_state"))))

    # Formato más amigable
    .withColumn("seller_city_clean", initcap(col("seller_city")))

    # Tipos de datos
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("int"))

    # Manejo de nulos
    .fillna({
        "seller_city": "unknown",
        "seller_state": "NA"
    })

    # Filtrar registros inválidos
    .filter(col("seller_id").isNotNull())

    # Deduplicación
    .dropDuplicates(["seller_id"])
)

# Crear la tabla Silver de olist_sellers

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_sellers")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_sellers